# Trabalho Pratico 01 - Classificacao KNN
**GCC 128 - Inteligencia Artificial - UFLA**
Profs. Ahmed Ali Abdalla Esmin / Anna Paula Figueiredo

Este notebook implementa o algoritmo **K-Nearest Neighbors do zero (hardcore)** e o compara
com a implementacao da biblioteca **Scikit-learn**, usando a base de dados **Iris**.

Roteiro:
1. Carga e exploracao da base Iris
2. Particao treino/teste estratificada (implementada do zero)
3. KNN hardcore + metricas de avaliacao implementadas manualmente
4. Taxa de reconhecimento para k = {1, 3, 5, 7}
5. KNN com Scikit-learn
6. Matrizes de confusao e metricas (precisao, revocacao, acuracia)
7. Analise de desempenho: metricas, tempo de execucao e memoria
8. Conclusao

## 1. Importacoes e carga da base

In [1]:
import os
import sys
import time
import timeit
import tracemalloc
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "src")
%matplotlib inline

np.set_printoptions(precision=4, suppress=True)

In [2]:
from utils import (
    COLUNAS_ATRIBUTOS, COLUNA_ALVO, VALORES_K,
    carregar_iris, dividir_treino_teste,
)

df = carregar_iris()
print("Dimensoes da base:", df.shape)
df.head()

Dimensoes da base: (150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [3]:
print("Distribuicao das classes:")
print(df[COLUNA_ALVO].value_counts(), "\n")
print("Estatisticas descritivas:")
df[COLUNAS_ATRIBUTOS].describe()

Distribuicao das classes:
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64 

Estatisticas descritivas:


,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


A base Iris tem 150 amostras, 4 atributos numericos e 3 classes perfeitamente
balanceadas (50 amostras cada). Nao ha valores faltantes.

In [4]:
# Visualizacao: petalas separam as classes muito melhor que sepalas
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))
cores = {"setosa": "#4C72B0", "versicolor": "#DD8452", "virginica": "#55A868"}

for especie, grupo in df.groupby(COLUNA_ALVO):
    eixos[0].scatter(grupo["sepal_length"], grupo["sepal_width"],
                     label=especie, alpha=0.75, color=cores[especie])
    eixos[1].scatter(grupo["petal_length"], grupo["petal_width"],
                     label=especie, alpha=0.75, color=cores[especie])

eixos[0].set(xlabel="sepal_length", ylabel="sepal_width", title="Sepalas")
eixos[1].set(xlabel="petal_length", ylabel="petal_width", title="Petalas")
for eixo in eixos:
    eixo.legend()
    eixo.grid(alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\joaog\AppData\Local\Temp\ipykernel_1864\763841997.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


O grafico das petalas ja antecipa o resultado do classificador: **setosa** e
linearmente separavel das demais, enquanto **versicolor** e **virginica** se
sobrepoem na fronteira -- e exatamente ali que os erros do KNN vao aparecer.

## 2. Particao treino/teste estratificada

Implementada do zero em `src/utils.py`. A estratificacao mantem a proporcao de
cada classe nos dois conjuntos, e a semente fixa garante que **os dois
classificadores recebam exatamente a mesma particao** -- condicao necessaria
para a comparacao ser justa.

In [5]:
X = df[COLUNAS_ATRIBUTOS].values
y = df[COLUNA_ALVO].values
classes = sorted(np.unique(y).tolist())

X_treino, X_teste, y_treino, y_teste = dividir_treino_teste(X, y)

print(f"Treino: {X_treino.shape[0]} amostras | Teste: {X_teste.shape[0]} amostras")
print("\nDistribuicao no treino:", dict(zip(*np.unique(y_treino, return_counts=True))))
print("Distribuicao no teste :", dict(zip(*np.unique(y_teste,  return_counts=True))))

Treino: 105 amostras | Teste: 45 amostras

Distribuicao no treino: {'setosa': np.int64(35), 'versicolor': np.int64(35), 'virginica': np.int64(35)}
Distribuicao no teste : {'setosa': np.int64(15), 'versicolor': np.int64(15), 'virginica': np.int64(15)}


## 3. O algoritmo KNN implementado do zero

O KNN e um *lazy learner*: nao ha fase de treino propriamente dita, apenas o
armazenamento da base. Toda a decisao acontece na classificacao:

1. calcular a distancia da amostra a **todos** os pontos de treino;
2. selecionar os **k** pontos mais proximos;
3. atribuir a classe majoritaria entre esses vizinhos.

Distancia euclidiana:  $d(a,b) = \sqrt{\sum_i (a_i - b_i)^2}$

O codigo completo esta em `src/knn_hardcore.py`. Abaixo, os pontos centrais.

In [6]:
from knn_hardcore import KNNHardcore, relatorio_classificacao
import inspect

# Nucleo do algoritmo: classificacao de uma unica amostra
print(inspect.getsource(KNNHardcore._classificar_amostra))

    def _classificar_amostra(self, amostra):
        """Classifica uma unica amostra pela votacao dos k vizinhos."""
        # 1) distancia da amostra para todos os pontos de treino
        distancias = self._funcao_distancia(amostra, self.X_treino)

        # 2) indices dos k menores valores (argpartition e O(n), mais barato
        #    que ordenar tudo); depois ordenamos apenas esses k
        idx_parciais = np.argpartition(distancias, self.k - 1)[: self.k]
        idx_vizinhos = idx_parciais[np.argsort(distancias[idx_parciais])]

        rotulos_vizinhos = self.y_treino[idx_vizinhos]

        # 3) votacao majoritaria simples
        votos = Counter(rotulos_vizinhos)
        maior = max(votos.values())
        empatados = [rotulo for rotulo, n in votos.items() if n == maior]

        if len(empatados) == 1:
            return empatados[0]

        # Desempate: vence a classe cujo vizinho mais proximo aparece antes
        # na lista ja ordenada por distancia.
        for rotulo in r

Dois detalhes de implementacao que valem destaque:

- **`np.argpartition`** encontra os k menores em tempo O(n), em vez de ordenar
  todas as n distancias em O(n log n). So os k selecionados sao ordenados depois.
- **Desempate**: quando duas classes empatam em votos, vence aquela cujo vizinho
  mais proximo aparece primeiro na lista ordenada por distancia -- o mesmo
  criterio adotado pelo Scikit-learn, o que permite a comparacao direta.

## 4. Metricas de avaliacao implementadas manualmente

Todas derivadas da matriz de confusao, sem usar `sklearn.metrics`:

| Metrica | Formula |
|---|---|
| Acuracia | acertos / total |
| Precisao (classe c) | $VP_c / (VP_c + FP_c)$ |
| Revocacao (classe c) | $VP_c / (VP_c + FN_c)$ |
| F1-score (classe c) | $2PR/(P+R)$ |

In [7]:
from knn_hardcore import matriz_confusao, acuracia, metricas_por_classe
from utils import imprimir_matriz_confusao, imprimir_relatorio

print(inspect.getsource(metricas_por_classe))

def metricas_por_classe(y_true, y_pred, classes):
    """Precisao, revocacao e F1 por classe, calculados da matriz de confusao.

    precisao_c  = VP_c / (VP_c + FP_c)   -> coluna c da matriz
    revocacao_c = VP_c / (VP_c + FN_c)   -> linha  c da matriz
    f1_c        = 2 * P * R / (P + R)
    """
    matriz = matriz_confusao(y_true, y_pred, classes)
    resultado = {}
    for i, classe in enumerate(classes):
        vp = matriz[i, i]
        fp = matriz[:, i].sum() - vp
        fn = matriz[i, :].sum() - vp
        precisao = vp / (vp + fp) if (vp + fp) > 0 else 0.0
        revocacao = vp / (vp + fn) if (vp + fn) > 0 else 0.0
        f1 = (
            2 * precisao * revocacao / (precisao + revocacao)
            if (precisao + revocacao) > 0
            else 0.0
        )
        resultado[classe] = {
            "precisao": float(precisao),
            "revocacao": float(revocacao),
            "f1": float(f1),
            "suporte": int(matriz[i, :].sum()),
        }
    return re

## 5. Taxa de reconhecimento para k = {1, 3, 5, 7}

Item explicitamente pedido no enunciado.

In [8]:
acuracias_hc = {}
relatorios_hc = {}

for k in VALORES_K:
    modelo = KNNHardcore(k=k, metrica="euclidiana")
    modelo.fit(X_treino, y_treino)
    y_pred = modelo.predict(X_teste)

    relatorios_hc[k] = relatorio_classificacao(y_teste, y_pred, classes)
    acuracias_hc[k] = relatorios_hc[k]["acuracia"]

print("TAXA DE RECONHECIMENTO - KNN HARDCORE\n")
for k, valor in acuracias_hc.items():
    acertos = int(round(valor * len(y_teste)))
    print(f"  k = {k}:  {valor:.4f}  ({acertos}/{len(y_teste)} amostras)")

TAXA DE RECONHECIMENTO - KNN HARDCORE

  k = 1:  0.9333  (42/45 amostras)
  k = 3:  0.9333  (42/45 amostras)
  k = 5:  0.9556  (43/45 amostras)
  k = 7:  0.9556  (43/45 amostras)


In [9]:
plt.figure(figsize=(5.5, 3.8))
barras = plt.bar([str(k) for k in acuracias_hc], list(acuracias_hc.values()),
                 color="#4C72B0", width=0.55)
for barra, valor in zip(barras, acuracias_hc.values()):
    plt.text(barra.get_x() + barra.get_width()/2, valor + 0.02,
             f"{valor:.4f}", ha="center", fontsize=9)
plt.ylim(0, 1.08)
plt.xlabel("k (numero de vizinhos)")
plt.ylabel("Taxa de reconhecimento")
plt.title("Taxa de reconhecimento por k - KNN hardcore")
plt.grid(axis="y", alpha=0.3)
plt.gca().set_axisbelow(True)
plt.tight_layout()
plt.show()

C:\Users\joaog\AppData\Local\Temp\ipykernel_1864\2987655498.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.1 Matrizes de confusao e metricas detalhadas (hardcore)

In [10]:
for k in VALORES_K:
    print("=" * 62)
    print(f"KNN HARDCORE - k = {k}")
    print("=" * 62)
    imprimir_matriz_confusao(relatorios_hc[k]["matriz_confusao"], classes)
    print()
    imprimir_relatorio(relatorios_hc[k], classes)
    print()

KNN HARDCORE - k = 1
  (linha = real | coluna = predito)
                  setosa  versicolor   virginica
      setosa          15           0           0
  versicolor           0          14           1
   virginica           0           2          13

      classe    precisao   revocacao    f1-score   suporte
      setosa      1.0000      1.0000      1.0000        15
  versicolor      0.8750      0.9333      0.9032        15
   virginica      0.9286      0.8667      0.8966        15
 media macro      0.9345      0.9333      0.9333
 media pond.      0.9345      0.9333      0.9333

  Acuracia (taxa de reconhecimento): 0.9333

KNN HARDCORE - k = 3
  (linha = real | coluna = predito)
                  setosa  versicolor   virginica
      setosa          15           0           0
  versicolor           0          14           1
   virginica           0           2          13

      classe    precisao   revocacao    f1-score   suporte
      setosa      1.0000      1.0000      1.0000     

## 6. Classificador com Scikit-learn

Agora a mesma tarefa usando `KNeighborsClassifier`, com a **mesma particao** e a
**mesma metrica de distancia**.

In [11]:
from knn_sklearn import treinar_e_prever, relatorio_classificacao_sklearn

acuracias_sk = {}
relatorios_sk = {}

for k in VALORES_K:
    _, y_pred_sk = treinar_e_prever(X_treino, y_treino, X_teste, k)
    relatorios_sk[k] = relatorio_classificacao_sklearn(y_teste, y_pred_sk, classes)
    acuracias_sk[k] = relatorios_sk[k]["acuracia"]

print("TAXA DE RECONHECIMENTO - KNN SKLEARN\n")
for k, valor in acuracias_sk.items():
    acertos = int(round(valor * len(y_teste)))
    print(f"  k = {k}:  {valor:.4f}  ({acertos}/{len(y_teste)} amostras)")

TAXA DE RECONHECIMENTO - KNN SKLEARN

  k = 1:  0.9333  (42/45 amostras)
  k = 3:  0.9333  (42/45 amostras)
  k = 5:  0.9556  (43/45 amostras)
  k = 7:  0.9556  (43/45 amostras)


In [12]:
for k in VALORES_K:
    print("=" * 62)
    print(f"KNN SKLEARN - k = {k}")
    print("=" * 62)
    imprimir_matriz_confusao(relatorios_sk[k]["matriz_confusao"], classes)
    print()
    imprimir_relatorio(relatorios_sk[k], classes)
    print()

KNN SKLEARN - k = 1
  (linha = real | coluna = predito)
                  setosa  versicolor   virginica
      setosa          15           0           0
  versicolor           0          14           1
   virginica           0           2          13

      classe    precisao   revocacao    f1-score   suporte
      setosa      1.0000      1.0000      1.0000        15
  versicolor      0.8750      0.9333      0.9032        15
   virginica      0.9286      0.8667      0.8966        15
 media macro      0.9345      0.9333      0.9333
 media pond.      0.9345      0.9333      0.9333

  Acuracia (taxa de reconhecimento): 0.9333

KNN SKLEARN - k = 3
  (linha = real | coluna = predito)
                  setosa  versicolor   virginica
      setosa          15           0           0
  versicolor           0          14           1
   virginica           0           2          13

      classe    precisao   revocacao    f1-score   suporte
      setosa      1.0000      1.0000      1.0000       

### 6.1 Matrizes de confusao lado a lado

In [13]:
fig, eixos = plt.subplots(2, 4, figsize=(17, 8))

for coluna, k in enumerate(VALORES_K):
    for linha, (rotulo, relatorios, cmap) in enumerate(
        [("hardcore", relatorios_hc, "Blues"), ("sklearn", relatorios_sk, "Greens")]
    ):
        matriz = relatorios[k]["matriz_confusao"]
        eixo = eixos[linha, coluna]
        eixo.imshow(matriz, interpolation="nearest", cmap=cmap)
        eixo.set_title(f"{rotulo} - k={k}", fontsize=10)
        eixo.set_xticks(range(len(classes)))
        eixo.set_yticks(range(len(classes)))
        eixo.set_xticklabels([c[:4] for c in classes], fontsize=8)
        eixo.set_yticklabels([c[:4] for c in classes], fontsize=8)
        if coluna == 0:
            eixo.set_ylabel("Real", fontsize=9)
        if linha == 1:
            eixo.set_xlabel("Predito", fontsize=9)
        limite = matriz.max() / 2.0
        for i in range(len(classes)):
            for j in range(len(classes)):
                eixo.text(j, i, matriz[i, j], ha="center", va="center", fontsize=11,
                          color="white" if matriz[i, j] > limite else "black")

plt.suptitle("Matrizes de confusao: hardcore (azul) x sklearn (verde)", fontsize=13)
plt.tight_layout()
plt.show()

C:\Users\joaog\AppData\Local\Temp\ipykernel_1864\1306615366.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Analise de desempenho

### 7.1 Comparacao das metricas de avaliacao

In [14]:
linhas = []
for k in VALORES_K:
    hc, sk = relatorios_hc[k], relatorios_sk[k]
    linhas.append({
        "k": k,
        "acuracia_HC": hc["acuracia"],   "acuracia_SK": sk["acuracia"],
        "precisao_HC": hc["macro"]["precisao"],  "precisao_SK": sk["macro"]["precisao"],
        "revocacao_HC": hc["macro"]["revocacao"], "revocacao_SK": sk["macro"]["revocacao"],
        "f1_HC": hc["macro"]["f1"],      "f1_SK": sk["macro"]["f1"],
    })

tabela = pd.DataFrame(linhas).set_index("k")
print("Metricas (medias macro) - HC = hardcore, SK = sklearn\n")
display(tabela.round(4))

diferenca_max = max(
    abs(tabela[f"{m}_HC"] - tabela[f"{m}_SK"]).max()
    for m in ["acuracia", "precisao", "revocacao", "f1"]
)
print(f"\nMaior diferenca absoluta entre as implementacoes: {diferenca_max:.2e}")

Metricas (medias macro) - HC = hardcore, SK = sklearn



,acuracia_HC,acuracia_SK,precisao_HC,precisao_SK,revocacao_HC,revocacao_SK,f1_HC,f1_SK
k,,,,,,,,
1,0.9333,0.9333,0.9345,0.9345,0.9333,0.9333,0.9333,0.9333
3,0.9333,0.9333,0.9345,0.9345,0.9333,0.9333,0.9333,0.9333
5,0.9556,0.9556,0.9556,0.9556,0.9556,0.9556,0.9556,0.9556
7,0.9556,0.9556,0.9556,0.9556,0.9556,0.9556,0.9556,0.9556



Maior diferenca absoluta entre as implementacoes: 0.00e+00


In [15]:
# As predicoes sao identicas amostra a amostra?
print("Concordancia entre as predicoes:\n")
for k in VALORES_K:
    pred_hc = KNNHardcore(k=k).fit(X_treino, y_treino).predict(X_teste)
    _, pred_sk = treinar_e_prever(X_treino, y_treino, X_teste, k)
    iguais = int(np.sum(pred_hc == pred_sk))
    print(f"  k={k}: {iguais}/{len(y_teste)} identicas ({100*iguais/len(y_teste):.2f}%)")

Concordancia entre as predicoes:

  k=1: 45/45 identicas (100.00%)
  k=3: 45/45 identicas (100.00%)
  k=5: 45/45 identicas (100.00%)
  k=7: 45/45 identicas (100.00%)


As duas implementacoes produzem **exatamente os mesmos resultados** -- nao apenas
metricas agregadas equivalentes, mas a mesma predicao para cada amostra. Isso
valida a implementacao hardcore: qualquer divergencia indicaria erro na logica de
distancia, na selecao dos vizinhos ou no criterio de desempate.

### 7.2 Tempo de execucao

In [16]:
def medir_tempo(funcao, repeticoes=7, numero=20):
    """Melhor de N execucoes -- estimativa menos sujeita a ruido do SO."""
    return min(timeit.repeat(funcao, number=numero, repeat=repeticoes)) / numero

tempos = []
for k in VALORES_K:
    def rodar_hc(k=k):
        modelo = KNNHardcore(k=k); modelo.fit(X_treino, y_treino)
        return modelo.predict(X_teste)
    def rodar_sk(k=k):
        return treinar_e_prever(X_treino, y_treino, X_teste, k)[1]

    t_hc = medir_tempo(rodar_hc) * 1000
    t_sk = medir_tempo(rodar_sk) * 1000
    tempos.append({"k": k, "hardcore_ms": t_hc, "sklearn_ms": t_sk,
                   "razao_HC/SK": t_hc / t_sk})

tabela_tempo = pd.DataFrame(tempos).set_index("k")
print("Tempo de fit + predict (melhor de 7 x 20 execucoes)\n")
display(tabela_tempo.round(3))

Tempo de fit + predict (melhor de 7 x 20 execucoes)



,hardcore_ms,sklearn_ms,razao_HC/SK
k,,,
1,0.649,1.082,0.600
3,0.671,1.102,0.609
5,0.687,1.106,0.621
7,0.693,1.110,0.624


Resultado contraintuitivo: na base Iris a implementacao **hardcore chega a ser
mais rapida** que a do Scikit-learn. A razao nao e que o laco em Python seja
eficiente, e sim que a base e pequena demais (105 amostras de treino): o tempo do
sklearn e dominado por **overhead fixo** -- validacao de entrada, checagem de
parametros e construcao da estrutura de busca (KD-tree) -- que nao chega a ser
amortizado. A secao seguinte mostra o que acontece quando a base cresce.

### 7.3 Uso de memoria

In [17]:
def medir_memoria(funcao):
    tracemalloc.start()
    funcao()
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return pico / 1024.0

memorias = []
for k in VALORES_K:
    def rodar_hc(k=k):
        modelo = KNNHardcore(k=k); modelo.fit(X_treino, y_treino)
        return modelo.predict(X_teste)
    def rodar_sk(k=k):
        return treinar_e_prever(X_treino, y_treino, X_teste, k)[1]
    memorias.append({"k": k,
                     "hardcore_KB": medir_memoria(rodar_hc),
                     "sklearn_KB": medir_memoria(rodar_sk)})

print("Pico de memoria alocada durante fit + predict\n")
display(pd.DataFrame(memorias).set_index("k").round(1))

Pico de memoria alocada durante fit + predict



,hardcore_KB,sklearn_KB
k,,
1,10.1,26.4
3,8.8,29.6
5,8.8,33.1
7,8.8,37.0


A versao hardcore usa menos memoria porque guarda apenas as matrizes de treino e
um vetor de distancias por amostra. O sklearn aloca, alem dos dados, a estrutura
de indexacao e os metadados do objeto estimador.

### 7.4 Escalabilidade: onde o sklearn compensa

O experimento abaixo repete a medicao de tempo variando o tamanho da base, com
dados sinteticos de mesma dimensionalidade (4 atributos).

In [18]:
rng = np.random.default_rng(0)
tamanhos = [150, 500, 1000, 2500, 5000, 10000]
escala = []

for n in tamanhos:
    n_teste = max(1, n // 5)
    Xb = rng.normal(size=(n, 4))
    yb = rng.choice(["a", "b", "c"], size=n)
    Xq = rng.normal(size=(n_teste, 4))

    def rodar_hc():
        modelo = KNNHardcore(k=5); modelo.fit(Xb, yb); return modelo.predict(Xq)
    def rodar_sk():
        return treinar_e_prever(Xb, yb, Xq, 5)[1]

    t_hc = min(timeit.repeat(rodar_hc, number=1, repeat=3)) * 1000
    t_sk = min(timeit.repeat(rodar_sk, number=1, repeat=3)) * 1000
    escala.append({"n_treino": n, "n_teste": n_teste,
                   "hardcore_ms": t_hc, "sklearn_ms": t_sk,
                   "razao_HC/SK": t_hc / t_sk})

tabela_escala = pd.DataFrame(escala).set_index("n_treino")
display(tabela_escala.round(3))

,n_teste,hardcore_ms,sklearn_ms,razao_HC/SK
n_treino,,,,
150,30,0.798,1.163,0.686
500,100,2.553,1.480,1.725
1000,200,7.015,1.990,3.524
2500,500,30.878,4.053,7.619
5000,1000,105.286,7.552,13.941
10000,2000,385.195,20.065,19.197


In [19]:
plt.figure(figsize=(6.5, 4.4))
plt.plot(tabela_escala.index, tabela_escala["hardcore_ms"], "o-",
         label="KNN hardcore", color="#4C72B0")
plt.plot(tabela_escala.index, tabela_escala["sklearn_ms"], "s-",
         label="KNN sklearn", color="#55A868")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Numero de amostras de treino")
plt.ylabel("Tempo de fit + predict (ms)")
plt.title("Escalabilidade das implementacoes (k=5)")
plt.grid(alpha=0.3, which="both")
plt.legend()
plt.tight_layout()
plt.show()

C:\Users\joaog\AppData\Local\Temp\ipykernel_1864\771775751.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


O cruzamento das curvas resume a comparacao: a versao hardcore vence apenas no
regime de base muito pequena. A partir de algumas centenas de amostras o sklearn
assume a lideranca e a distancia cresce rapidamente -- na maior base testada ele
e mais de uma ordem de grandeza mais rapido. A implementacao hardcore e O(n) por
consulta em Python puro; o sklearn usa KD-tree e codigo compilado.

## 8. Conclusao

**Qualidade da classificacao.** As duas implementacoes obtiveram resultados
**identicos** para todos os valores de k testados -- mesmas predicoes, mesmas
matrizes de confusao, mesmas metricas. Isso era o esperado: o KNN e um algoritmo
deterministico, e com a mesma particao, a mesma distancia e o mesmo criterio de
desempate nao ha espaco para divergencia. A coincidencia funciona como
**validacao da implementacao propria**.

**Efeito de k.** A taxa de reconhecimento subiu de 93,33% (k=1 e k=3) para
95,56% (k=5 e k=7). Valores maiores de k suavizam a fronteira de decisao e
reduzem a sensibilidade a ruido; com k=1 a classificacao depende de um unico
vizinho, o que torna o modelo instavel perto da fronteira.

**Erros cometidos.** Todos os erros ocorreram entre *versicolor* e *virginica* --
as duas classes que se sobrepoem no espaco de atributos, como o grafico das
petalas ja indicava. A classe *setosa* foi classificada perfeitamente
(precisao = revocacao = 1,00) em todas as configuracoes, por ser linearmente
separavel das demais.

**Desempenho computacional.** Na base Iris a implementacao hardcore foi cerca de
**1,6x mais rapida** e usou **3 a 4x menos memoria** que a do sklearn. Esse
resultado, porem, nao generaliza: ele se explica pelo overhead fixo do sklearn
(validacao e construcao de KD-tree) sobre uma base de apenas 105 amostras de
treino. O experimento de escalabilidade mostra a inversao -- a partir de ~500
amostras o sklearn ja e mais rapido, e com 10.000 amostras a vantagem passa de
**20x**.

**Conclusao geral.** Para fins didaticos e bases pequenas, a implementacao
propria e perfeitamente adequada e tem a vantagem de tornar o algoritmo
transparente. Para uso em producao, a biblioteca e a escolha correta: alem da
escalabilidade muito superior, oferece tratamento de casos de borda,
paralelizacao, multiplas metricas de distancia e estruturas de indexacao
otimizadas -- tudo isso ja testado e mantido.